# 블랙박스 야간 위험성 평가 AI — 최종 시스템 설명서

**프로젝트**: DLthon 2026-04-22 ~ 2026-04-26 (Motorcycle Night Ride Segmentation)  
**컨셉**: 한문철 TV + 스카우터 — 사후 분석용 위험도 자동 판정  
**작성일**: 2026-04-24 (실험·실습 종료 시점)  
**위치**: `archive/final_report/MODEL_OVERVIEW.ipynb`

---

## 한 줄 요약

> **U-Net R34 (Lane Focus recipe) 5-fold ensemble + Apple Depth Pro + 4-Zone Ego Corridor → 4 경고 모듈 (FCW/LDW/BSW/HEAD), bin = max(severities). 도메인 테스트 9/10 exact, OOD SUV 4 안전 → 치명 정상 감지.**

## 핵심 숫자

| 지표 | 값 |
|---|---|
| Segmentation mIoU(fg) | **0.7845 ± 0.0100** (5-fold) |
| Lane Mark IoU | **0.3611 ± 0.0245** (5-fold, baseline 대비 ×360) |
| 도메인 10장 위험도 정확 일치 | **9/10** (lenient 10/10) |
| OOD SUV 사고 프레임 | 4점 안전 → **치명** 정상 감지 |
| 거리 추정 | metric (Apple Depth Pro 1m 단위) |
| VP 추정 성공률 | 100% (Method L, 평균 conf 0.83) |

![최종 HUD 시스템 — 4 bin 대표 (안전/주의/위험/치명) + Before/After ("한문철 TV + 스카우터" 컨셉)](../hud_samples.png)

*최종 HUD 시스템 — 4 bin 대표 (안전/주의/위험/치명) + Before/After ("한문철 TV + 스카우터" 컨셉)*

---

# 1. 시스템 전체 파이프라인

단일 프레임을 입력받아 위험 등급(안전/주의/위험/치명)과 4개 경고 신호를 출력합니다.

```
                          [입력 RGB 프레임]
                                 │
                                 ▼
  ┌──────────────────────────────────────────────────────────────┐
  │  (1) Segmentation                                            │
  │      U-Net R34 + Lane Focus recipe (5-fold ensemble)         │
  │      → 7-class mask (Background / Undrivable / Road /        │
  │         Lane Mark / Moveable / My bike / Rider)              │
  └──────────────────────────────────────────────────────────────┘
                                 │
         ┌───────────────────────┼───────────────────────┐
         ▼                       ▼                       ▼
  ┌─────────────┐         ┌─────────────┐         ┌─────────────┐
  │ (2) Lane    │         │ (3) Object  │         │ (4) Depth   │
  │   Lines     │         │   Extract   │         │   Apple     │
  │  (Hough+    │         │   (CC on    │         │   Depth Pro │
  │   cluster)  │         │    Moveable │         │   metric+   │
  │             │         │    /Bike/   │         │   focal)    │
  │             │         │    Rider)   │         │             │
  └─────────────┘         └─────────────┘         └─────────────┘
         │                       │                       │
         ▼                       ▼                       │
  ┌─────────────┐         ┌─────────────┐                │
  │ (5) VP      │         │ (6) Ego     │                │
  │  Method L   │         │   Mask      │                │
  │ (lane 쌍    │         │  (closest   │                │
  │  교점 median│         │   CC to     │                │
  │  + conf)    │         │   bottom)   │                │
  └─────────────┘         └─────────────┘                │
         │                       │                       │
         └───────────┬───────────┘                       │
                     ▼                                   │
         ┌─────────────────────┐                         │
         │ (7) Distance        │◄────────────────────────┘
         │     Estimator       │
         │   d_ground (bbox y) │
         │   d_size  (bbox w)  │
         │   d_depth (Depth Pro)│
         │   → fused (median   │
         │     filtered + W/M) │
         └─────────────────────┘
                     │
                     ▼
         ┌─────────────────────┐
         │ (8) Zone Corridor   │
         │   critical/danger/  │
         │   caution + side-   │
         │   lobe L/R          │
         └─────────────────────┘
                     │
                     ▼
  ┌──────────────────────────────────────────────────────────────┐
  │  (9) 4 Warning Modules (각 Level 0~3)                        │
  │      FCW  (Forward Collision)   ── 정면 가장 가까운 객체     │
  │      LDW  (Lane Departure)      ── ego 가 ego_lane 이탈      │
  │      BSW  (Blind Spot)          ── 측면 + 침범                │
  │      HEAD (Head-on)             ── 정면 역주행 (cut-in 분리) │
  │                                                              │
  │  최종 bin = max(FCW, LDW, BSW, HEAD)                         │
  │      L0=안전 / L1=주의 / L2=위험 / L3=치명                   │
  └──────────────────────────────────────────────────────────────┘
                                 │
                                 ▼
                          [HUD 출력 + JSON]
```

**핵심 설계 원칙**:
1. **합산 점수가 아닌 max severity** — "하나라도 치명이면 치명". 약한 신호 합산으로 위험을 만들지 않음.
2. **Hybrid 거리 추정** — 야간 모션블러 단일 추정기 신뢰 X. 3-estimator 가중 + outlier reject.
3. **Zone confirmation** — 점수 양산하지 않음. 거리·기하 신호가 zone 안에 들어와야 confirmed.
4. **Cut-in vs Head-on 구분** — 옆에서 끼어드는 SUV (BSW) 와 정면 역주행 (HEAD) 을 분리.

---

# 2. 데이터

## 2.1 데이터셋

- **출처**: Kaggle — Motorcycle Night Ride Segmentation
- **이미지 수**: 200장 (단일 영상에서 추출, 1280×720 부근)
- **레이블**: 픽셀 단위 7-class semantic segmentation mask
- **분할**: Train 170 / Test 30 holdout, **5-fold CV** on train

## 2.2 클래스 매핑

| ID | Class | 비율 (대략) | 비고 |
|---|---|---|---|
| 0 | Background | 0% | 사용 안 함 (loss weight 0) |
| 1 | Undrivable | 큼 | 인도/벽/하늘 등 비주행 영역 |
| 2 | Road | 큼 | 주행 가능 노면 |
| 3 | Lane Mark | **0.6%** | 차선 페인트 (극소수, 핵심 병목) |
| 4 | Moveable | 중 | 다른 차/오토바이/사람 (위협 대상) |
| 5 | My bike | 작음 | ego 차량 본체 |
| 6 | Rider | 작음 | ego 운전자 (헬멧/팔) |

## 2.3 도메인 특성

- **야간 위주** (저조도, 모션블러, 헤드라이트 글레어)
- **POV = 오토바이 블랙박스** (헬멧/핸들바 화면 일부 가림)
- 일반적인 자율주행 데이터셋(Cityscapes 주간 자동차 POV)과 도메인 차이 큼 → pretrain 전이 효과 한정 (§3.1 참조)

![클래스별 픽셀 분포 — Lane Mark 0.6% 가 두드러진 minority](../eda_class_distribution.png)

*클래스별 픽셀 분포 — Lane Mark 0.6% 가 두드러진 minority*

![데이터셋 sanity check — 원본·GT mask·class 매핑](../dataset_sanity_check.png)

*데이터셋 sanity check — 원본·GT mask·class 매핑*

---

# 3. Segmentation 모델 — Lane Focus Recipe

## 3.1 모델 구조

| 항목 | 값 |
|---|---|
| Architecture | **U-Net** |
| Encoder | **ResNet-34** (24M params) |
| Encoder weights | **ImageNet** pretrained |
| Output channels | 7 (NUM_CLASSES) |
| Library | `segmentation_models_pytorch` (smp) |

**왜 ResNet-34 / U-Net?** — Pretrain ablation 결과 SegFormer-B0 (3.7M, Cityscapes) 보다 R34 (24M, ImageNet) 가 +3%p. 야간 moto POV → 주간 car POV 도메인 갭이 Cityscapes feature 의 전이 이득을 상쇄. **용량/아키텍처가 pretrain 소스보다 중요** (DAY2_SUMMARY §2.2).

## 3.2 학습 하이퍼파라미터 (확정값)

| 항목 | 값 | 비고 |
|---|---|---|
| **입력 해상도** | **768 × 768** | thin Lane Mark 보존 위해 default 512 → 768 상향 |
| **Batch size** | 4 | 768² + R34 → VRAM 한계 |
| **Epochs** | 40 | Cosine decay 마지막까지 |
| **Optimizer** | AdamW | weight_decay=1e-4 |
| **Learning rate** | 1e-4 (init) | Cosine annealing → 0 |
| **Scheduler** | `CosineAnnealingLR(T_max=40)` |  |
| **Seed** | 42 | random/np/torch 동시 고정 |
| **NUM_WORKERS** | 0 | AIFFEL JupyterHub shm 제한 호환 |

## 3.3 Loss — Combined (Focal + Dice + WeightedCE)

Lane Mark 0.6% 극소수 → 단일 loss 로는 무시됨. **3-loss combined** 로 해결:

```python
class_weights = torch.tensor(
    [0.0, 0.5, 0.5, 50.0, 2.0, 0.5, 1.0],  # idx 3 = Lane Mark
    dtype=torch.float32).to(device)

focal_fn = smp.losses.FocalLoss(mode='multiclass', gamma=2.0)
dice_fn  = smp.losses.DiceLoss(mode='multiclass', from_logits=True)
wce_fn   = nn.CrossEntropyLoss(weight=class_weights)

def combined_loss(logits, targets):
    return 0.3 * focal_fn(logits, targets) \
         + 0.3 * dice_fn(logits, targets) \
         + 0.4 * wce_fn(logits, targets)
```

**가중치 의도**:
- `Background = 0.0` → 학습 신호에서 완전 배제 (mIoU(fg) 평가)
- `Lane Mark = 50.0` → 0.6% 픽셀 보상 (다른 class 대비 100배)
- `Moveable = 2.0` → 위협 대상이라 보강
- 나머지는 0.5~1.0 (균형)

**3-loss 조합 의도**:
- **Focal (γ=2.0, w=0.3)**: easy negative 억제 (Background 픽셀 압도적)
- **Dice (w=0.3)**: 작은 영역 IoU 직접 최적화
- **WeightedCE (w=0.4)**: per-pixel 분포 보정 (Lane Mark 50× 가중)

## 3.4 Lane Mark Morphology Dilation (★ 핵심 트릭)

Lane Mark GT 가 1~2 픽셀 두께라 학습 신호 자체가 약함. **train mask 만 dilation 5×5 로 두께화**:

```python
# train transform 안에서 (val 은 dilation X)
lane = (mask == 3).astype(np.uint8)
kernel = np.ones((5, 5), np.uint8)
dilated = cv2.dilate(lane, kernel, iterations=1)
expand_into = ((mask == 0) | (mask == 2))   # Background/Road 만 잠식
mask[(dilated == 1) & expand_into] = 3
```

**효과**: Lane Mark IoU **0.0001 → 0.3611** (×360 배). val mask 는 원본 그대로라 metric 오염 없음.

## 3.5 Augmentation

```python
A.Compose([
    A.Resize(768, 768),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.15,
                               contrast_limit=0.15, p=0.5),
    A.CLAHE(clip_limit=3.0, p=0.3),
    A.GaussNoise(var_limit=(5.0, 25.0), p=0.15),
    A.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]),
    ToTensorV2(),
])
```

야간 도메인이라 **HSV/색상 증강은 의도적으로 제외** (헤드라이트 색감 왜곡 방지). 밝기·노이즈 위주.

## 3.6 5-Fold Cross Validation 결과 (★ 최종 숫자)

| Fold | mIoU(fg) | Lane IoU | 학습 시간 |
|---|---|---|---|
| 0 | 0.7927 | 0.3615 | 14.3 분 |
| 1 | **0.7959** ⭐ | 0.3311 | 15.9 분 |
| 2 | 0.7810 | **0.4001** ⭐ | 13.4 분 |
| 3 | 0.7852 | 0.3730 | 14.3 분 |
| 4 | 0.7676 | 0.3399 | 14.9 분 |
| **평균 ± std** | **0.7845 ± 0.0100** | **0.3611 ± 0.0245** | 14.6 분 |

**Per-class IoU (fold 0 마지막 epoch)**:

| Class | IoU |
|---|---|
| Background | (loss 0, 평가 제외) |
| Undrivable | 0.9308 |
| Road | 0.8391 |
| **Lane Mark** | **0.3503** |
| Moveable | 0.7617 |
| My bike | 0.9388 |
| Rider | 0.9099 |

**Baseline 대비 개선**:
- mIoU(fg): 0.6948 (baseline U-Net R34 + CE) → 0.7845 (**+8.97%p**)
- Lane IoU: 0.0001 → 0.3611 (**×3611 배**)

## 3.7 Inference 시 Ensemble

최종 inference 는 **5 fold majority vote per pixel**:

```python
# _v5_warnings.py — predict_seg_ensemble()
votes = np.zeros((NUM_CLASSES, H, W), dtype=np.int32)
for model in models:
    mask = predict_seg(model, img_np, device)
    for c in range(NUM_CLASSES):
        votes[c] += (mask == c).astype(np.int32)
return np.argmax(votes, axis=0).astype(np.uint8)
```

단일 fold 보다 boundary 가 안정적 (특히 Lane Mark thin region).

## 3.8 체크포인트 파일

| 파일 | 용도 |
|---|---|
| `lane_focus_fold0_best.pth` | fold 0 best (mIoU 0.7927) |
| `lane_focus_fold1_best.pth` | fold 1 best (mIoU 0.7959 — 발표용 단일 후보) |
| `lane_focus_fold2_best.pth` | fold 2 best (Lane IoU 0.4001 — 발표용 단일 후보) |
| `lane_focus_fold3_best.pth` | fold 3 best |
| `lane_focus_fold4_best.pth` | fold 4 best |

각 ckpt 는 `{'state_dict', 'epoch', 'miou', 'lane_iou', 'fold'}` dict 구조.

![Pretrain ablation — "용량 > pretrain 소스" 의 정량 근거](../cityscapes_vs_imagenet.png)

*Pretrain ablation — "용량 > pretrain 소스" 의 정량 근거*

![Lv1 5-fold (Dice+CE) — Lane Mark 여전히 0% 인 병목 시각화](../lv1_kfold_comparison.png)

*Lv1 5-fold (Dice+CE) — Lane Mark 여전히 0% 인 병목 시각화*

![학습 곡선 — Baseline 기준 (Lane Focus 도 동일 패턴이지만 mIoU/Lane 모두 큰 폭 개선)](../baseline_curves.png)

*학습 곡선 — Baseline 기준 (Lane Focus 도 동일 패턴이지만 mIoU/Lane 모두 큰 폭 개선)*

---

# 4. 거리 추정 (Distance Estimator)

## 4.1 왜 Hybrid 인가

단일 추정기는 야간에 모두 한계가 있어 **3-estimator fusion** 으로 결정:

| 추정기 | 강점 | 약점 |
|---|---|---|
| **d_ground** (bbox 바닥 y) | 기하학적 정확 (flat ground 가정 시) | 차량 일부 가려지면 실패 |
| **d_size** (bbox 폭 + 차종 prior) | occlusion 내성 | 차종 분류 오류에 민감 |
| **d_depth** (Apple Depth Pro) | 픽셀 단위 metric | 야간 sky/원거리 noise |

**Depth Anything V2 단일 사용 실패 근거**: 도메인 200장 중 Moveable bbox 의 99% 를 "원거리" 로 판정 (가까운 차도 멀게 본다). 야간 inverse depth saturate. → metric depth 인 **Apple Depth Pro** 로 교체 (v5).

## 4.2 Focal Length 추정

두 가지 경로:

1. **Apple Depth Pro 자동 추정** (우선) — `post_process_depth_estimation` 출력의 `focal_length` 필드 사용
2. **차선폭 prior 역산** (fallback) — 한국 차선폭 3.5m, 카메라 높이 1.0m 가정으로 Mobileye Gen 1 방식

```python
# geometric_distance.py — estimate_focal_from_lanes()
# d ∝ 1/(y - vp_y), lane_width_px ∝ 1/d
# → const = lane_width_px * (y - vp_y) = f * LANE_WIDTH_M / CAMERA_HEIGHT_M
f_px = const * CAMERA_HEIGHT_M / LANE_WIDTH_M
```

Sanity: `0.3 * W < f_px < 2.5 * W` 범위 밖이면 fallback (`f = W * 1.2`).

## 4.3 3-Estimator 공식

### d_ground (bbox 바닥)
```
d = f_px × camera_h / (y_bottom - vp_y)
```

### d_size (bbox 폭 + class prior)
```
d = f_px × W_real / w_bbox_px

W_real (meter):  car=1.8, truck=2.5, bus=2.55,
                 motorcycle=0.8, bicycle=0.6, person=0.5
```

Sub-class 분류는 bbox aspect ratio 로 coarse:
- ar > 2.5 → truck/bus
- 1.2 < ar < 2.5 → car
- ar < 0.7 → motorcycle/person

### d_depth (Apple Depth Pro, metric)
```
patch = depth_map[y0:y1, x0:x1]
valid = patch[(0.3 < patch) & (patch < 100)]
d = median(valid)   # bbox 중심 대표값
```

## 4.4 Fusion (가중 + outlier reject)

```python
# Depth Pro (metric) 모드 가중치
weights = {'ground': 0.25, 'size': 0.25, 'depth': 0.50}

# 1. Outlier reject: median 의 1/3 ~ 3배 밖 estimator 제거
med = np.median(valid_estimates)
filtered = {k: v for k, v in valid.items()
            if 0.33 * med < v < 3.0 * med}

# 2. Filtered 만으로 weighted mean
d_fused = sum(filtered[k] * weights[k] / total_w for k in filtered)

# 3. Confidence
#    n>=3 estimators 동일 bin 합의 → 'high'
#    n>=2 fused bin 포함        → 'medium'
#    그 외                       → 'low'
```

**거리 bin** (warning module 임계값으로 사용):
- `<5m`, `5-15m`, `15-30m`, `>30m`

## 4.5 도메인 검증 결과

Apple Depth Pro 9/9 정확도 (사용자 라벨링 기준). DA V2 대비 야간 객체 거리 신뢰도 향상.

![dist_idx 시각화 — Bike/Rider 1.0, Moveable 분포 확인](../distance_index_samples.png)

*dist_idx 시각화 — Bike/Rider 1.0, Moveable 분포 확인*

![VP ↔ Depth 교차검증 (Pearson r = +0.741) — VP primary, Depth confirmation](../vp_depth_crosscheck.png)

*VP ↔ Depth 교차검증 (Pearson r = +0.741) — VP primary, Depth confirmation*

---

# 5. Vanishing Point & Ego Mask

## 5.1 Lane Line Detection

Lane Mark 픽셀 → 직선 추출:

```python
# _v33_hierarchical.py — detect_lane_lines()
lane_binary = (mask == 3).astype(np.uint8) * 255
lane_clean = cv2.morphologyEx(lane_binary, cv2.MORPH_OPEN, np.ones((3,3)))
lines_raw = cv2.HoughLinesP(
    lane_clean, rho=1, theta=np.pi/180,
    threshold=40, minLineLength=30, maxLineGap=25)
# (slope, x_mid) greedy 클러스터링
# → 각 cluster 의 length 가중 평균으로 unified line
```

출력 line 구조:
```python
{'dydx': float, 'c': float,    # x = dydx * y + c
 'y_top': int, 'y_bot': int,   # 유효 y 범위
 'length': float}              # 픽셀 길이
```

## 5.2 VP — Method L (★ 100% 성공)

**핵심 가설**: "정상 운전자는 차선을 지킨다". → 좌/우 차선의 교점 = ego 진행 방향.

```python
# vp_from_lanes() — pairwise 교점의 length-weighted median
for i, j in combinations(lines, 2):
    교점 (x, y) 계산, 이미지 bounds 체크
    weight = sqrt(length_i * length_j)
    inters.append((x, y, weight))
vx = weighted_median(inters_x)
vy = weighted_median(inters_y)
confidence = 1 - std_x / W   # 흩어짐 적을수록 높음
```

**도메인 10장 결과**: 10/10 성공, 평균 confidence 0.83 (range 0.65~0.99).

**Fallback 우선순위**:
1. Method L (lane pairwise) — conf > 0.3
2. Method C (Road 좌/우 경계 polyfit) — Day 2 까지 primary 였으나 v5 에서 backup
3. 이미지 중앙 (W/2, H*0.4) — 위 둘 모두 실패 시

## 5.3 Ego Mask 추출 (★ False Positive 방어)

**문제**: My bike (class 5) / Rider (class 6) 옆부분이 Moveable (class 4) 로 오분류 → 자기 자신을 위협으로 착각 (FCW 오발화).

**해결**: 중앙 하단 anchor 기준 closest CC + dilation 흡수.

```python
# _v5_warnings.py — get_ego_mask()
bike_base = ((mask == 5) | (mask == 6)).astype(np.uint8)
closed = cv2.morphologyEx(bike_base, cv2.MORPH_CLOSE, np.ones((5,5)))
dilated = cv2.dilate(closed, np.ones((5,5)), iterations=1)  # 5px 흡수
n_cc, labels = cv2.connectedComponents(dilated)

# (W/2, H) 에 가장 가까운 centroid 의 CC 만 ego
anchor = np.array([W/2, H])
best_cc = argmin(distance(centroid_k, anchor) for k)
ego_mask = (labels == best_cc)
```

**Filter 규칙**: Moveable bbox 가 ego_mask 와 IoU > 0.40 (큰 bbox 는 0.55) 이면 제거.

## 5.4 Ego X (LDW 입력)

LDW 는 "내가 ego_lane 중심에서 얼마나 벗어났나" 인데, **이미지 중앙 ≠ 내 위치**. → ego_mask centroid x 를 실제 ego_x 로 사용.

```python
ys_em, xs_em = np.where(ego_mask > 0)
ego_cx = float(xs_em.mean())   # 진짜 my bike 중심
```

![Method L 파이프라인 — Hough lane 추출 → pairwise 교점 → length-weighted median](../method_l_pipeline.png)

*Method L 파이프라인 — Hough lane 추출 → pairwise 교점 → length-weighted median*

![(참고) Method C — Road 좌/우 경계 polyfit 결과 5샘플. v5 backup 으로 사용](../vp_method_c_5samples.png)

*(참고) Method C — Road 좌/우 경계 polyfit 결과 5샘플. v5 backup 으로 사용*

---

# 6. 4-Zone Ego Corridor

## 6.1 동기

거리·기하 신호만으로 점수를 합산하면 **약한 신호 누적으로 위험을 만들어내는** 부작용 발생 (v3.x). → **Zone confirmation layer** 도입: 위협 객체가 "내 경로" 안 어디에 있는가를 별도로 본다.

## 6.2 4 Zone 정의

VP 와 ego_lane 하단 (Lx, Rx) 으로 사다리꼴 corridor 구성. 안쪽일수록 심각:

| Zone | alpha (VP→bottom 비율) | severity | 의미 |
|---|---|---|---|
| `caution` | 0.15 ~ 1.0 | 1 | 전체 ego corridor (lane band) |
| `danger` | 0.55 ~ 1.0 | 2 | 하위 45% (FCW L2 거리 수준) |
| `critical` | 0.85 ~ 1.0 | 3 | 하위 15% (바로 앞 치명) |
| `sidelobe_left/right` | 좌/우 30% 확장 | (BSW용) | 측면 위협 별도 신호 |

**Sidelobe 폭**: `lane_width_at_bottom * 0.30` 만큼 좌/우 확장.

```python
# ego_corridor.py — build_corridor()
def trapezoid(alpha_top):
    yt = vp_y + alpha_top * (H - 1 - vp_y)
    xtl = lane_x_at(vp_x, vp_y, Lx, H, yt)   # 좌선 보간
    xtr = lane_x_at(vp_x, vp_y, Rx, H, yt)   # 우선 보간
    return [(xtl, yt), (xtr, yt), (Rx, H-1), (Lx, H-1)]
```

## 6.3 Bbox → Zone 분류

**Anchor**: bbox bottom-center (= 노면 접지점).

```python
# 1. Anchor 가 어느 zone 에 있는지 (inner 부터)
for name in ('critical', 'danger', 'caution'):
    if mask[name][foot_y, foot_x] > 0:
        anchor_zone = name; break

# 2. Soft promote: 더 안쪽 zone 에 30%+ overlap 이면 승급
if bbox_ratio['critical'] > 0.30: sev = max(sev, 3)
elif bbox_ratio['danger']   > 0.30: sev = max(sev, 2)
elif bbox_ratio['caution']  > 0.30: sev = max(sev, 1)

# 3. Sidelobe (BSW 별도 신호): 40%+ 면 hit
if max(sl_left, sl_right) > 0.40:
    sidelobe_hit = 'left' or 'right'
```

## 6.4 Safeguards

- **VP confidence < 0.3** → corridor = `None`. Zone 기능 비활성.
- **Lane width < 20px** (뒤집힘) → `None`.
- LDW 활성 frame 에서는 호출측이 zone clamp 비활성화 (갓길 차량을 안전으로 잘못 분류 방지).

![Safe Corridor — 도메인 10장에 critical/danger/caution + sidelobe 시각화](../safe_corridor_domain10.png)

*Safe Corridor — 도메인 10장에 critical/danger/caution + sidelobe 시각화*

![4-Zone 분류 — bbox 가 어느 zone 에 anchored 되는지](../risk_zones_domain10.png)

*4-Zone 분류 — bbox 가 어느 zone 에 anchored 되는지*

---

# 7. 위험도 시스템 v5 — 4 경고 모듈

## 7.1 v3.x/v4 와의 근본 차이

| 항목 | v2.6 / v3.x | **v5 (최종)** |
|---|---|---|
| 점수 합산 | ✓ (9 flag × weight) | ✗ |
| 거리 단위 | 픽셀 dist_idx (0~1) | metric (m) |
| Bin 결정 | total ≤ 4/10/16 thresholds | **max(severity)** |
| 모듈 수 | 9~11 flag | **4 warning** (FCW/LDW/BSW/HEAD) |

**왜 max?** — 합산은 "약한 위반 N개 = 강한 위반 1개" 등식을 만든다. 운전 안전 도메인에선 **하나라도 critical 이면 critical**.

## 7.2 4 Warning Module

각 모듈은 Level 0~3 (없음/주의/위험/치명) 출력. 최종 bin = `max(FCW.level, LDW.level, BSW.level, HEAD.level)`.

### FCW (Forward Collision Warning) — 정면 충돌

정면 (`0.35 ≤ cx/W ≤ 0.65`) 객체 중 가장 가까운 거리 기준:

| 거리 | bbox area ratio | Level |
|---|---|---|
| `< 3.0m` | OR `> 25%` | **3 (치명)** |
| `< 5.0m` | — | 2 (위험) |
| `< 10.0m` | — | 1 (주의) |
| 그 외 | — | 0 |

### LDW (Lane Departure Warning) — ego 차선 이탈

ego_x (실제 my bike 중심) vs ego_lane 중심선 거리 / lane width:

```python
lateral_ratio = abs(ego_x - lane_center) / lane_width
```

| `lateral_ratio` | Level | 비고 |
|---|---|---|
| `> 0.7` | **0 (측정 불가)** | seg 실패 가능 → guard |
| `> 0.4` 또는 lane 밖 | 2 (위험, 이탈) |  |
| `> 0.2` | 1 (주의, 편향) |  |
| 그 외 | 0 |  |

### BSW (Blind Spot Warning) — 측면 위협

좌측 (`cx/W < 0.35`) / 우측 (`> 0.65`) 가장 가까운 객체 + Zone sidelobe hit:

| 조건 | Level |
|---|---|
| `< 3m` AND lane 침범 | **3 (치명)** |
| `< 3m` (침범 없어도) | 2 (위험) |
| `< 5m` | 1 (주의) |
| 그 외 | 0 |

**Sidelobe boost** (Frame 11 fix): 거리 추정 실패 (None) 라도 sidelobe ratio > 0.45 이면 BSW L2 로 promote.

### HEAD (Head-on) — 정면 역주행

**Cut-in (옆에서 끼어듦) 과 분리**:

```python
is_frontal_center = (0.40 <= cx_rel <= 0.60)   # 중앙 20%
is_frontal_aspect = (aspect = w/h >= 1.8)       # 넓고 낮음 (정면 차)
if not (is_frontal_center and is_frontal_aspect):
    return Level 0   # cut-in → BSW 로만

# 정면 역주행 확정 후 거리 기준
if d < 3.0m: Level 3
elif d < 5.0m: Level 2
```

## 7.3 Zone Clamp Patches (v5.x 안정화)

**Patch 1 — FCW L3 zone clamp** (Frame 6 fix):  
`area_ratio > 25%` 단독으로 L3 받은 경우, foot point 가 danger zone 밖이면 L2 로 강등. 단, BSW L2+ 동시 활성 (cut-in 시나리오) 이면 L3 유지.

**Patch 2 — BSW sidelobe boost** (Frame 11 fix):  
거리 추정 실패해서 BSW L0 인데 sidelobe ratio > 0.45 면 L2 로 promote ("옆에서 들어오는 게 분명").

## 7.4 최종 bin → 색상

| bin | severity | color | 의미 |
|---|---|---|---|
| 안전 | 0 | `#2ECC71` 초록 | 모든 warning Level 0 |
| 주의 | 1 | `#F39C12` 주황 | 어떤 warning 이라도 L1 |
| 위험 | 2 | `#E74C3C` 빨강 | 어떤 warning 이라도 L2 |
| 치명 | 3 | `#8E44AD` 보라 | 어떤 warning 이라도 L3 |

## 7.5 Warning 색상 (HUD bbox)

| Warning | Color | 우선순위 (겹침 시) |
|---|---|---|
| HEAD | `#DA70D6` 보라 | 1 (최고) |
| FCW | `#FF4444` 빨강 | 2 |
| BSW | `#FF8C00` 주황 | 3 |
| LDW | `#FFD700` 노랑 (lane band) | 4 (bbox 안 그림) |

![v5 도메인 10장 HUD — 4 warning 모듈 결과. Exact 9/10, Lenient 10/10](../v5_gallery.png)

*v5 도메인 10장 HUD — 4 warning 모듈 결과. Exact 9/10, Lenient 10/10*

---

# 8. (참고) 위험도 시스템 진화 과정

최종 v5 가 어떻게 도달했는지 흐름. **운영 시점엔 v5 만 사용**.

| 버전 | 날짜 | 핵심 변화 | OOD SUV (사고 프레임) |
|---|---|---|---|
| v1 | 04-22 | 6 flag (정면/측면/교차/경로/다수/저조도), score sum | 4 [안전] ❌ |
| v2 | 04-22 | 9 flag + 4 bin, weighted sum | 4 [안전] ❌ |
| v2.5 | 04-23 | + Depth Tier (DA V2 p90 기반 4단계) | 9 [주의] |
| v2.6 | 04-23 | intrusion 룰 개선 (wider band, bbox y 샘플링) | 14 [위험] |
| v2.6 + Lane Focus | 04-23 | 새 모델로 mask 품질 개선 | **20 [치명]** ✓ |
| v3.2 Complete | 04-23 | Method L VP + Lane violation strict + X1 fallback | **20 [치명]** (VIOL_x1) |
| v4 (distance-based) | 04-23 | DA V2 → metric 거리 fit, score sum 유지 | 19 [위험] |
| **v5** (Warning-based) | **04-24** | **4 모듈 + max severity + Apple Depth Pro + Zone Corridor** | **치명** ✓ |

**핵심 통찰 (얻은 순서대로)**:
1. cone-based intrusion 은 false positive 90% — 정상 주행 차도 발화 (v2 → v3)
2. Lane Mark 기반 line 교차 (strict) 가 의미 있음 (v3.2)
3. SUV 가 lane 가림 → strict 안 됨 → X1 fallback (이미지 중앙 가로지름) 도입 (v3.2)
4. DA V2 inverse depth 야간 saturate → metric depth 필요 (v4 → v5)
5. **점수 합산은 의미 왜곡** — max severity 로 전환 (v4 → v5)
6. **Cut-in vs Head-on 분리** 필요 (v5 HEAD 모듈)
7. Zone confirmation 으로 false positive 추가 억제 (v5)

![v1 — OOD SUV 4점 안전 ❌](../ood_test_suv_result.png)

*v1 — OOD SUV 4점 안전 ❌*

![v2.5 — OOD SUV 9점 주의 (depth tier 추가)](../ood_test_v25_result.png)

*v2.5 — OOD SUV 9점 주의 (depth tier 추가)*

![v2.6 — OOD SUV 14점 위험 (intrusion 룰 개선)](../ood_test_v26_result.png)

*v2.6 — OOD SUV 14점 위험 (intrusion 룰 개선)*

![Lane Focus + v2.6 — OOD SUV 20점 치명 ✓ (모델 개선이 결정타)](../new_model_ood_result.png)

*Lane Focus + v2.6 — OOD SUV 20점 치명 ✓ (모델 개선이 결정타)*

---

# 9. 최종 도메인 테스트 결과 (v5)

사용자 직접 캡처 10장 (주간 서울 도심, 일부 위험/사고 프레임 포함). GT 는 사용자 라벨링.

## 9.1 정확도

- **Exact match: 9 / 10** (단일 차이도 "위험 ↔ 치명" 같은 인접 bin)
- **Lenient match (인접 bin 허용): 10 / 10**
- Bin 분포: 안전 5 / 주의 0 / 위험 2 / 치명 3 (한 프레임 GT 와 lenient 매치)

## 9.2 프레임별 상세

| # | Frame | bin | GT | 활성 Warning | d_front (m) | d_left | d_right |
|---|---|---|---|---|---|---|---|
| 1 | 114941 | 주의 | 주의 | FCW | 9.34 | - | - |
| 2 | 115220 | 주의 | 주의 | FCW, LDW | 5.49 | - | 10.13 |
| 3 | 115945 | 위험 | 위험 | LDW, BSW | 29.22 | 3.95 | - |
| 4 | 120038 | 위험 | 주의~위험 | (lenient) | - | - | - |
| 5 | **120122 (SUV)** | **치명** | 치명 | FCW, LDW, BSW | 3.93 | 1.95 | - |
| 6 | 141655 | 주의 | 위험 | FCW | 7.17 | - | 41.82 |
| 7 | 141745 | 주의 | 주의 | FCW, BSW | 7.18 | 3.65 | 15.4 |
| 8 | 141904 | 안전 | 주의 | LDW | 18.94 | 7.05 | 5.75 |
| 9 | 142019 | 치명 | 치명 | FCW, LDW, BSW | 4.62 | 6.0 | 6.07 |
| 10 | 144602 | 치명 | 치명 | FCW, LDW, BSW | 2.04 | 2.3 | 2.48 |

## 9.3 OOD SUV 핵심 검증 (Frame 5)

사용자가 직접 캡처한 "SUV 가 lane 가로지르며 끼어드는 사고 직전" 프레임:

| 시스템 버전 | Score / Bin | 감지 경로 |
|---|---|---|
| v2 (baseline) | 4 / **안전** ❌ | 놓침 |
| v2.6 + Lane Focus | 20 / 치명 | depth tier + intrusion 합산 |
| v3.2 Complete | 20 / 치명 | VIOL_x1 (이미지 중앙 가로지름) |
| **v5 (최종)** | — / **치명** ✓ | **FCW (3.9m) + LDW (좌측 이탈) + BSW (좌 1.95m)** |

v5 는 Score 합산이 아니라 **세 가지 독립 경고가 동시에 켜져서** 치명 판정 — 더 설명력 있는 신호.

![v5 — Test set 추가 평가 갤러리 (holdout 일부)](../v5_testset_gallery.png)

*v5 — Test set 추가 평가 갤러리 (holdout 일부)*

---

# 10. 재사용 모듈 — `risk_assessor.py`

팀 배포·외부 사용을 위한 self-contained API. **단, v2.6 시점 인터페이스** (max-severity v5 는 `_v5_warnings.py` 에 prototype 로 존재).

## 10.1 기본 사용법

```python
from risk_assessor import RiskAssessor

# (1) 단일 모델 + Depth Anything V2 자동 로드
assessor = RiskAssessor(
    seg_checkpoint='lane_focus_fold0_best.pth',
    use_depth=True,
    device='cuda',
)

# (2) 단일 이미지 평가
result = assessor.assess('frame.png')
print(f"Score: {result['score']} / 23")
print(f"Bin  : {result['bin']}")
print(f"Flags: {[k for k,v in result['flags'].items() if v]}")

# (3) HUD 시각화 저장
assessor.draw_hud_to_file('frame.png', result, 'hud_output.png')
```

## 10.2 외부 mask 주입 (다른 아키텍처와 호환)

```python
# 팀원이 자기 모델로 segment 한 mask 사용
assessor = RiskAssessor(seg_checkpoint=None, use_depth=True)
result = assessor.assess_with_mask(
    image='frame.png',
    mask=my_mask_HxW_uint8,    # 7-class
    depth_map=my_depth,         # optional
)
```

## 10.3 결과 dict 구조

```python
{
    'score': int,         # 0~23 (v2.6 기준)
    'bin': str,           # '안전' | '주의' | '위험' | '치명'
    'v2_score': int,      # 9 flag 합산 (0~21)
    'depth_score': int,   # Depth Tier (0/1/3/5)
    'flags': {            # 9 boolean flag
        'intrusion', 'cross', 'frontal_near', 'side_near',
        'narrow_road', 'space_asym', 'dense', 'low_light', 'asymmetry',
    },
    'threat_directions': ['L', 'C', 'R'],   # 위협 방향
    'vp': {'x', 'y', 'source'},              # vanishing point
    'objects': [...],                         # detected moveables
    'image_shape': (H, W),
}
```

## 10.4 9 Flag 가중치 (v2.6)

| Flag | 가중치 | 한글 |
|---|---|---|
| intrusion | 5 | 차선 침범 (중과실) |
| cross | 4 | 정면 교차 (LTAP) |
| frontal_near | 3 | 정면 근접 |
| side_near | 3 | 측면 근접 |
| asymmetry | 3 | 주행 비대칭 |
| narrow_road | 2 | 도로 협소 |
| space_asym | 1 | 좌우 비대칭 |
| dense | 1 | 다수 차량 |
| low_light | 1 | 저조도 |

Depth Tier 보너스 (`p90 / 255`): `≥0.7 → +5`, `≥0.5 → +3`, `≥0.3 → +1`.  
Bin 임계: `≤4 안전`, `≤10 주의`, `≤16 위험`, `>16 치명`.

## 10.5 v5 인터페이스 (4 warning) 사용

현재 `risk_assessor.py` 는 v2.6 까지. v5 (max-severity) 는 다음 함수 직접 호출:

```python
from _v5_warnings import (
    load_seg_ensemble, predict_seg_ensemble,
    load_depth_pipe, predict_depth, assess_v5,
)

models = load_seg_ensemble([
    'lane_focus_fold0_best.pth',
    'lane_focus_fold1_best.pth',
    'lane_focus_fold2_best.pth',
    'lane_focus_fold3_best.pth',
    'lane_focus_fold4_best.pth',
])
depth_pipe = load_depth_pipe('cuda')

import numpy as np; from PIL import Image
img = np.array(Image.open('frame.png').convert('RGB'))
mask = predict_seg_ensemble(models, img, 'cuda')
depth = predict_depth(depth_pipe, img)

result = assess_v5(img, mask, depth, img_name='frame.png')
print(result['bin'], result['active_warnings'])
```

(Day 3 후속: `risk_assessor.py` 클래스에 v5 모드 옵션 추가 검토)

---

# 11. 환경 / 의존성

## 11.1 학습 환경

**환경 A (로컬, 코드 작성·디버깅)**:
- GPU: RTX 4070 Laptop (8GB VRAM)
- OS: Windows 11
- Python 3.10+

**환경 B (AIFFEL JupyterHub, 5-fold 학습)**:
- GPU: Tesla T4 (15GB VRAM)
- OS: Linux (Kubernetes 컨테이너)
- PyTorch 2.7.1 + CUDA 12.2
- `NUM_WORKERS=0` 필수 (shm 제한)
- 작업 디렉토리: `~/work/dacon_block` (`/work` 외 세션 간 비유지)

## 11.2 패키지

```bash
pip install \
    torch torchvision \
    segmentation-models-pytorch \
    transformers \
    albumentations \
    opencv-python \
    pillow matplotlib \
    numpy
```

**모델 weights** (자동 다운로드):
- ImageNet pretrained ResNet-34 (smp 자동)
- Apple Depth Pro: `apple/DepthPro-hf` (HuggingFace, fp16, ~1GB)
- (legacy) Depth Anything V2: `depth-anything/Depth-Anything-V2-Small-hf`

---

# 12. 한계와 후속 작업

## 12.1 의도적 한계 (정직성)

1. **Temporal 정보 없음** — 단일 프레임 분석. 충돌 시점·속도·궤적 예측 불가.
2. **Lane Mark IoU 0.36** — 5-fold ensemble 으로 boundary 안정화했지만 절대값은 여전히 낮음.
3. **데이터 200장 단일 영상** — 일반화 주장 제한적. (도메인 OOD 10장은 사용자 직접 캡처 보충)
4. **AI 과실 비율 판정 안 함** — 법적 영역 회피.
5. **사후 분석 도구** — 실시간 예측 시스템 아님.

## 12.2 모델 측면

- 야간 헤드라이트 글레어가 강한 프레임에서 Moveable boundary 흐려짐
- 5-fold ensemble 추론은 단일 fold 대비 ~5배 시간 (각 모델 별도 forward) — 실시간 사용 시 단일 fold 또는 weight average 권장
- Apple Depth Pro fp16 메모리 ~3GB, 추론 시간 frame 당 ~0.5초 (RTX 4070)

## 12.3 위험도 측면

- HEAD 모듈은 "넓은 정면 차량" 만 감지 — 측면 충돌 직전 cut-in 은 BSW 로 분류 (의도적)
- LDW guard (`lateral_ratio > 0.7` 측정 불가) 가 진짜 큰 이탈을 놓칠 수 있음 — fallback 필요 시 ego_mask 신뢰도 같이 체크
- Zone Corridor 는 VP confidence < 0.3 면 전체 비활성 → corridor-less mode 결과 보정 필요

## 12.4 가능한 후속

1. **Temporal smoothing** (5 프레임 EMA over warning levels) → flicker 감소
2. **Test 30장 holdout 평가** — 전체 metric 확정
3. **CAM 시각화** — 모델이 어디 보고 판단하는지 (브리핑 TODO)
4. **유튜브 사고 영상 실증** (2~3건) — 외부 OOD 추가
5. `risk_assessor.py` 에 v5 모드 정식 통합 (현재 `_v5_warnings.py` prototype)
6. Lane Mark IoU 추가 개선 — Boundary Loss / RMI Loss 시도

---

# 13. 핵심 파일 인벤토리

최종 시스템을 구성하는 파일들 (`archive/` 기준 상대 경로).

## 13.1 코드

| 파일 | 역할 |
|---|---|
| `05_dataset.py` | Dataset / split / class mapping |
| `_resume_5fold.py` | 5-fold 학습 entry (Lane Focus recipe) |
| `risk_assessor.py` | **공식 API (v2.6)** — 외부 배포용 |
| `_v5_warnings.py` | **v5 prototype** — 4 warning 모듈 |
| `_v33_hierarchical.py` | 공통 함수 (seg/depth 로드, lane detection) |
| `geometric_distance.py` | 거리 추정기 (3-estimator fusion) |
| `ego_corridor.py` | 4-zone corridor 생성·분류 |

## 13.2 모델 체크포인트

| 파일 | 크기 | 용도 |
|---|---|---|
| `lane_focus_fold0_best.pth` | ~93MB | fold 0 |
| `lane_focus_fold1_best.pth` | ~93MB | fold 1 (mIoU 최고) |
| `lane_focus_fold2_best.pth` | ~93MB | fold 2 (Lane IoU 최고) |
| `lane_focus_fold3_best.pth` | ~93MB | fold 3 |
| `lane_focus_fold4_best.pth` | ~93MB | fold 4 |
| `baseline_best.pth` | ~93MB | (참고) 초기 baseline |

## 13.3 결과 / 시각화

| 파일 | 내용 |
|---|---|
| `lane_focus_5fold_results.json` | 5-fold 학습 history + 최종 metric |
| `v5_results.json` | 도메인 10장 v5 평가 결과 |
| `v5_gallery.png` | 도메인 10장 v5 HUD 시각화 |
| `v5_individual/` | 프레임별 단일 HUD PNG |
| `hud_samples.png` | 발표용 HUD 4 bin 대표 + Before/After |
| `lv1_kfold_comparison.png` | baseline 대비 fold-wise 비교 |
| `cityscapes_vs_imagenet.png` | Pretrain ablation |

## 13.4 문서

| 파일 | 내용 |
|---|---|
| `DLthon.md` | 초기 브리핑 |
| `PROJECT_MEMORY.md` | 누적 결정 기록 |
| `DAY2_SUMMARY.md` | Day 2 종합 |
| `AUTONOMOUS_SESSION_SUMMARY.md` | v3.2 세션 요약 |
| `V33_SUMMARY.md`, `V4_REPORT.md` | 위험도 진화 기록 |
| `PRESENTATION_REPORT.md` / `.ipynb` | 발표 자료 (이미지 포함) |
| `final_report/MODEL_OVERVIEW.ipynb` | **이 문서 — 최종 모델 설명서** |

---

# 14. 부록 — 한 페이지 요약 (cheat sheet)

**Segmentation**:  
U-Net + ResNet-34 (ImageNet) | 768×768 | BS=4 | AdamW lr=1e-4 wd=1e-4 | Cosine 40ep | seed=42  
Loss = 0.3·Focal(γ=2) + 0.3·Dice + 0.4·WCE(weight[3]=50)  
Train mask: Lane dilation 5×5 (val 원본)  
Aug: HFlip, BC(0.15), CLAHE(3.0), GaussNoise — HSV 없음 (야간)  
**5-fold mIoU(fg) 0.7845±0.0100, Lane IoU 0.3611±0.0245**  
Inference: **5-fold majority vote** per pixel

**Distance**:  
Apple Depth Pro (metric m + focal 자동) → fallback `f = W*1.2`  
3-estimator: ground (`f·h/(y_b-vp_y)`) + size (`f·W_real/w_px`) + depth (median)  
Fusion weight (metric): 0.25/0.25/**0.50** | outlier reject (3× median)

**VP**:  
Method L — lane pairwise 교점 length-weighted median, conf = 1 - std_x/W  
Fallback: Method C (Road edges) → image center

**Ego Mask**:  
(class 5 OR 6) → close 5×5 → dilate 5×5 → CC closest to (W/2, H)  
Filter: Moveable bbox IoU > 0.40 with ego_mask → 제거

**Zone Corridor**:  
alpha = {critical: 0.85, danger: 0.55, caution: 0.15} | sidelobe = lane_w·0.30  
Anchor = bbox bottom-center | Soft promote 30% | Sidelobe hit 40%

**Warnings (Level 0~3)**:
- **FCW** (정면): `<3m or area>25%` → L3, `<5m` → L2, `<10m` → L1
- **LDW** (이탈): lateral/lane_w `>0.4 or out` → L2, `>0.2` → L1, `>0.7` → L0 guard
- **BSW** (측면): `<3m + 침범` → L3, `<3m` → L2, `<5m` → L1, sidelobe>0.45 → L2 boost
- **HEAD** (역주행): `0.4≤cx/W≤0.6 + aspect≥1.8` 만 발화, `<3m`→L3, `<5m`→L2

**Final**:  
`bin = max(FCW, LDW, BSW, HEAD)` → 안전(0)/주의(1)/위험(2)/치명(3)

**도메인 10장**: exact 9/10, lenient 10/10, OOD SUV 4안전→치명 ✓